# 01 · Define & Explore — affinity maturation + developability theory, CDR contacts, the mock hello-world

**Standard slot:** *define & explore.* **For Project 15 this means:** you are NOT designing an
antibody de novo — you are **lead-optimizing an EXISTING antibody**. Start from a *known*
antibody-antigen complex (with a **measured KD in the literature**), understand which CDR residues
contact the antigen, fix the metrics table, and run the **mock** affinity-maturation hello-world
end-to-end (D0).

Run `00_setup.ipynb` first in this session. Everything here runs with **no GPU** on the deterministic
`mock` backend — switch to the real backends (ESM-1v / AbLang / ProteinMPNN / AF2-Multimer) on Colab
in notebooks 02–04.

## The problem in one screen

**Affinity maturation, computationally.** A therapeutic-antibody lead usually *binds*, but not tightly
or cleanly enough. **Lead optimization** — raising affinity while keeping **developability** (no
aggregation, no chemical-liability hotspots, expressible, stable) — is slow and expensive in the lab.
The computational job is to propose a **small, testable set** of CDR mutations that are *likely* to
improve the antibody, so the wet lab tests ~10 variants instead of thousands.

**Why "existing antibody", not de novo.** You inherit a real paratope and a real, measured starting
affinity. Mutations are **edits** to that paratope:
- **Framework is FIXED.** Only **CDR** positions (the loops that contact antigen) are varied — that is
  what "maturation" means. Touching the framework risks folding/expression and humanness.
- **CDR3** dominates the paratope (longest, most diverse loop); CDR1/CDR2 contribute too.

**The two honesty rules for this whole project:**
1. **Never fabricate KD / ΔΔG / affinity numbers.** The deliverable is a **ranked, ordered** set of
   candidate mutations + the **experiment** (SPR/DSF) that would test them — not predicted constants.
   Every mock number here is a dimensionless **ranking score**, flagged `SYNTHETIC`.
2. **Most predicted affinity-improving mutations do NOT validate.** Output a SMALL ranked set with
   mandatory **controls** (WT baseline + a destabilizing decoy). Report the rate, not the cherry.

## The metrics table (what we will score and filter on)

| Metric | Range | Means | Does **not** mean | Used for |
|--------|-------|-------|-------------------|----------|
| ESM-1v Δ-log-likelihood | ~[-3,+3] | protein-LM favors the mutation over WT (> 0) | higher affinity / a ΔΔG | RANK single mutations |
| AbLang naturalness | 0–1 | antibody-specific "natural-looking" prior | low immunogenicity / affinity | sanity-check CDR sequences |
| pae_interaction | Å | AF2-Multimer confidence in the **interface** pose | binding/affinity | **pose maintenance** (≤ 12) |
| scRMSD (vs parent) | Å | variant Fv backbone vs the parent pose | binding | **pose maintenance** (≤ 3.0) |
| developability liabilities | count | CDR chemical-liability motifs (NG/DG, Met-ox, free Cys, sequon) | a real TAP verdict | triage **before** synthesis |

The `"antibody"` cutoffs (scRMSD ≤ 3.0, pLDDT ≥ 70, pae_interaction ≤ 12) come from the shared
`filtering_pipeline.DEFAULT_CUTOFFS["antibody"]`. **ESM-1v/AbLang scores RANK candidates; they are NOT
affinities. Developability here is a TEACHING HEURISTIC, not the validated tool (TAP/CamSol/real
deamidation predictors)** — see `MANUAL.md §2` and `maturation_tools.py`.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Pick your antibody-antigen complex (with a published KD)

The single most important input is a **real antibody-antigen complex from SAbDab** that has a
**measured KD reported in the literature** — that KD is your starting affinity, the baseline every
proposed mutation is measured against. The accession below is a **candidate placeholder — verify it on
SAbDab/RCSB in Week 1, and confirm a published KD exists for it.** Do not assert any KD value here.

In [ ]:
# --- Campaign definition (EDIT in Week 1 after choosing + verifying your complex) ---
# Pick a well-characterized therapeutic Fab-antigen complex from SAbDab WITH a published KD.
COMPLEX_PDB = "VERIFY_ON_SABDAB"   # e.g. a therapeutic Fab-antigen complex (candidate — verify; needs a published KD)
ANTIGEN = "ANTIGEN"                 # name your antigen once the complex is chosen
PARENT_KD_NOTE = ("the parent KD is the MEASURED literature value for your chosen complex — record the "
                  "exact value + its citation in your problem statement. Do NOT invent a number here.")

print("complex (candidate — verify on SAbDab/RCSB):", COMPLEX_PDB)
print("antigen :", ANTIGEN)
print("parent KD:", PARENT_KD_NOTE)

## The parent antibody + its CDRs

`maturation_tools.EXAMPLE_FRAMEWORK` + `example_parent_sequence()` are a **teaching placeholder** so the
plumbing runs anywhere. In Week 1 you **replace** them with the chains read off your verified complex,
and you derive the CDR boundaries with a real antibody numbering scheme (IMGT/Kabat/Chothia via ANARCI)
— do not eyeball them. The framework is **fixed**; the CDR spans below are the only positions you may
mutate.

In [ ]:
from maturation_tools import (EXAMPLE_FRAMEWORK, example_parent_sequence, cdr_positions)

parent = example_parent_sequence()      # EXAMPLE placeholder VH — replace with your real chain
spans = cdr_positions()
print("framework:", EXAMPLE_FRAMEWORK["name"], "(teaching placeholder — replace with your real framework)")
print("parent length:", len(parent), "aa")
print("CDR spans (0-based, half-open) — the ONLY mutable positions:")
for cdr, (a, b) in spans.items():
    print(f"  {cdr}: residues {a+1}-{b}  loop = {parent[a:b]}")

## Identify CDR contact residues (the maturation targets)

Affinity maturation focuses mutations on residues that **contact the antigen** (or support contacting
loops). On a real complex you compute the **paratope** = antibody residues within ~4–5 Å of any antigen
atom (Biopython `NeighborSearch` on your verified PDB). Here, with no structure loaded, we mark the CDR
spans as the candidate region and leave the real contact extraction as a TODO for notebook 02 / Week 4.
Targeting *contact* residues (not all CDR residues) keeps the candidate set small and physically
motivated.

In [ ]:
# On a real complex, replace this with a contact calculation from the PDB:
#   from Bio.PDB import PDBParser, NeighborSearch
#   parse COMPLEX_PDB -> antibody atoms + antigen atoms
#   paratope = {antibody residues with any atom within 4.5 A of any antigen atom}
#   intersect paratope with the CDR spans -> the CONTACT residues you prioritise.
# For the mock hello-world we treat the CDR spans as the candidate region.
contact_region = []
for cdr, (a, b) in spans.items():
    contact_region += list(range(a + 1, b + 1))   # 1-based positions
print("candidate (CDR) positions to consider for mutation:", contact_region)
print("TODO (Week 4): intersect with the real paratope (<=4.5 A contacts) from your verified complex.")

## Affinity-maturation hello-world (mock backend, no GPU)

Score a few single CDR mutations with the ESM-1v / AbLang proxies, assemble a tiny ranked candidate
set, then pose-check + developability-scan it. This proves the plumbing (score → rank → pose check →
liability scan → controls) before any GPU time in notebooks 02–04. **Every number below is a SYNTHETIC
ranking score — never a KD, never report it as a real affinity.**

In [ ]:
from maturation_tools import (score_single_mutations, assemble_candidate_set,
                              score_variants, make_controls)

# Score every single substitution at every CDR position (framework fixed), then rank a SMALL set.
singles = score_single_mutations(parent, tool="mock")
cand = assemble_candidate_set(singles, top_n=5)
score_variants(cand, tool="mock")        # fills af2 pose check + developability (SYNTHETIC)

print(f"scored {len(singles)} single CDR mutations; showing the top-5 RANKED candidates (NOT KDs):\n")
print(f"{'mutation':10s} {'CDR':5s} {'esm1v':>7s} {'ablang':>7s} {'pae':>5s} {'scrmsd':>7s} {'liab':>5s}")
for v in cand:
    print(f"{'+'.join(v.mutations):10s} {v.cdr:5s} {v.esm1v:>+7.3f} {v.ablang:>7.3f} "
          f"{v.pae_interaction:>5.1f} {v.scrmsd:>7.3f} {v.n_liabilities:>5d}")

ctrls = make_controls(parent, antigen=ANTIGEN)
print("\ncontrols (mandatory):", [c.design_id for c in ctrls])
print("synthetic:", cand[0].synthetic, "->", cand[0].notes[0] if cand[0].notes else "")

## Visualize the complex (py3Dmol)

Use this to eyeball your real antibody-antigen complex and its paratope once you have the verified PDB.
The mock backend writes no structure.

In [ ]:
import py3Dmol

def show_pdb(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after fetching your verified complex with data/download_data.py):
# show_pdb("data/inputs/<YOUR_COMPLEX>.pdb")
print("show_pdb(pdb_path) ready — use it on your verified antibody-antigen complex.")

## D0 checklist
- [ ] 1-page **problem statement**: the chosen **SAbDab complex** (verified accession), its **measured
      literature KD** (value + citation — not invented), the CDR contact residues you will target, and
      **measurable** success criteria.
- [ ] Verified the complex on SAbDab/RCSB; confirmed a **published KD** exists; derived CDR boundaries
      with a real numbering scheme (not the EXAMPLE placeholder).
- [ ] Metric table understood, including that ESM-1v/AbLang are **ranking** signals (not affinity) and
      developability here is a **heuristic**.
- [ ] Mock hello-world run; top-ranked single mutations + SYNTHETIC pose/liability metrics printed.
- [ ] `LOG.md` entry (seed, what you ran).

**Next:** `02_generate.ipynb` — the single-mutation scoring + ProteinMPNN CDR-redesign campaign.